# The 2020 Coalition and the Reservoir of 2024 Switchers

Notebook 12 used 39 panelists to infer that the 2024 switchers were 2016 non-voters pulled in by the 2020 surge, already immigration-conservative. The CES 2020 cross-section (61,000) can profile the surge cohort directly, and the CES 2024 cross-section (60,000) can trace switchers back to 2016 with 16,807 recalled-Biden voters instead of 39. Both are on disk.

**Questions.** Who entered the Democratic coalition in 2020? Were they culturally distinct? Where did the 2024 switchers actually come from — by 2016 vote, and by 2020 attitude?

**Inputs:** `data/raw/ces/CES20_Common_OUTPUT_vv.csv`, `data/raw/ces/CCES24_Common_OUTPUT_vv_topost_final.csv`

**Coding note.** In the 2020 file, respondents who chose "Hispanic" as their race (`race == 3`) were not asked the separate `hispanic` item; Hispanic = `race == 3` or `hispanic == 1`. In 2024 the race-3 respondents are already coded `hispanic == 1`. The 2020 wave asks Medicare-for-all (`CC20_327a`) and a $15 minimum wage (`CC20_350b`); 2024 does not.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
RAW_DIR = Path('../data/raw'); TABLES_DIR = Path('../outputs/tables')
def wmean(x, w):
    m = x.notna() & w.notna()
    return np.average(x[m], weights=w[m]) if m.any() else np.nan

## 1. CES 2020: the coalition by 2016 behavior

In [ ]:
cols = ['commonpostweight', 'commonweight', 'presvote16post', 'CC20_410', 'pid7', 'birthyr', 'educ', 'race', 'hispanic', 'faminc_new', 'CL_2020gvm',
        'CC20_327a', 'CC20_350b', 'CC20_331a', 'CC20_331b', 'CC20_331d', 'CC20_331e', 'CC20_332a', 'CC20_302']
d = pd.read_csv(RAW_DIR / 'ces' / 'CES20_Common_OUTPUT_vv.csv', usecols=cols, low_memory=False)
d['wt'] = d['commonpostweight'].fillna(d['commonweight'])
d['v16'] = d['presvote16post'].map({1: 'Clinton', 2: 'Trump', 3: 'other', 4: 'other', 5: 'other', 6: 'other', 7: 'nonvoter'})
d['v20'] = d['CC20_410'].map({1: 'Biden', 2: 'Trump'})
cons = lambda c, sup: (d[c] == (1 if sup else 2)).astype(float).where(d[c].notna())
d['imm_cons'] = pd.concat([cons('CC20_331a', False), cons('CC20_331b', True), cons('CC20_331d', True), cons('CC20_331e', True)], axis=1).mean(axis=1)
d['m4a'] = cons('CC20_327a', True); d['minwage15'] = cons('CC20_350b', True); d['abortion_cons'] = cons('CC20_332a', False)
d['econ_perception'] = 3 - d['CC20_302'].where(d['CC20_302'].between(1, 5))
d['age'] = 2020 - d['birthyr']; d['college'] = (d['educ'] >= 5).astype(float)
d['hisp'] = ((d['race'] == 3) | (d['hispanic'] == 1)).astype(float); d['white'] = ((d['race'] == 1) & (d['hisp'] == 0)).astype(float); d['black'] = ((d['race'] == 2) & (d['hisp'] == 0)).astype(float)
d['lowinc'] = (d['faminc_new'].where(d['faminc_new'] <= 16) <= 5).astype(float).where(d['faminc_new'] <= 16)
d['pid7c'] = d['pid7'].where(d['pid7'] <= 7); d['validated_2020'] = d['CL_2020gvm'].notna().astype(float)
e20 = d[d['v20'].notna() & d['v16'].notna()].copy(); e20['path'] = e20['v16'] + '->' + e20['v20']
W = e20['wt'].sum()
print(f'CES 2020 respondents with a 2020 major-party vote and recalled 2016 status: {len(e20):,}')
for s in ['Clinton', 'Trump', 'other', 'nonvoter']:
    g = e20[e20['v16'] == s]
    print(f'  2016 {s:9s} {g["wt"].sum() / W:>6.1%} of 2020 voters -> Biden {wmean((g["v20"] == "Biden").astype(float), g["wt"]):.1%}, Trump {wmean((g["v20"] == "Trump").astype(float), g["wt"]):.1%}')
b20 = e20[e20['v20'] == 'Biden']; t20 = e20[e20['v20'] == 'Trump']
print(f'\nBiden 2020 voters (N = {len(b20):,}): Clinton 2016 {wmean((b20["v16"] == "Clinton").astype(float), b20["wt"]):.1%}, non-voter 2016 {wmean((b20["v16"] == "nonvoter").astype(float), b20["wt"]):.1%} (N = {int((b20["v16"] == "nonvoter").sum()):,}), Trump 2016 {wmean((b20["v16"] == "Trump").astype(float), b20["wt"]):.1%} (N = {int((b20["v16"] == "Trump").sum()):,})')
print(f'Trump 2020 voters: non-voter 2016 {wmean((t20["v16"] == "nonvoter").astype(float), t20["wt"]):.1%}')

rows = []
for p in ['Clinton->Biden', 'nonvoter->Biden', 'Trump->Biden', 'Clinton->Trump', 'nonvoter->Trump', 'Trump->Trump']:
    s = e20[e20['path'] == p]
    rows.append({'path': p, 'N': len(s), **{v: wmean(s[v], s['wt']) for v in ['imm_cons', 'm4a', 'minwage15', 'abortion_cons', 'econ_perception', 'age', 'college', 'white', 'hisp', 'black', 'lowinc', 'pid7c', 'validated_2020']}})
prof20 = pd.DataFrame(rows).set_index('path')
print('\nProfile by 2016->2020 path (weighted):'); print(prof20.round(2).to_string())
print('\nImmigration items 2020, share conservative — Clinton->Biden / nonvoter->Biden / nonvoter->Trump / Trump->Trump:')
for c, lab, sup in [('CC20_331a', 'oppose legal status', False), ('CC20_331b', 'more border patrol', True), ('CC20_331d', 'cut legal immigration 50%', True), ('CC20_331e', 'wall + $25B', True)]:
    x = (e20[c] == (1 if sup else 2)).astype(float).where(e20[c].notna())
    print(f'  {lab:26s} ' + ' / '.join(f'{wmean(x[e20["path"] == p], e20.loc[e20["path"] == p, "wt"]):.0%}' for p in ['Clinton->Biden', 'nonvoter->Biden', 'nonvoter->Trump', 'Trump->Trump']))
print('\nThe surge cohort (non-voter 2016 -> Biden) was not culturally distinct: immigration index 0.21 vs 0.18 for Clinton->Biden.')
print('It was younger, poorer, less educated, more weakly attached (pid7 2.33 vs 1.76), and less likely to be validated (61% vs 77%).')

## 2. CES 2020: the immigration-conservative fifth of Biden's coalition

In [ ]:
b20 = b20.copy(); b20['immC'] = (b20['imm_cons'] >= 0.5).astype(float); b20['econL'] = ((b20['m4a'] == 1) | (b20['minwage15'] == 1)).astype(float)
rows = []
for s in ['Clinton', 'nonvoter', 'Trump', 'ALL']:
    g = b20 if s == 'ALL' else b20[b20['v16'] == s]; gi = g[g['immC'] == 1]
    rows.append({'2016_vote': s, 'N': len(g), 'share_imm_cons': wmean(g['immC'], g['wt']), 'share_econleft_and_immcons': wmean(g['econL'] * g['immC'], g['wt']),
                 'M4A_among_immcons': wmean(gi['m4a'], gi['wt']), 'minwage15_among_immcons': wmean(gi['minwage15'], gi['wt']), 'N_immcons': len(gi)})
res20 = pd.DataFrame(rows).set_index('2016_vote')
print('Biden 2020 voters: immigration-conservative share by 2016 status, and economic views of the immigration-conservative:')
print(res20.round(3).to_string())
sc = b20[b20['v16'] == 'nonvoter']; sci = sc[sc['immC'] == 1]; scl = sc[sc['immC'] == 0]
rows = []
for v in ['hisp', 'white', 'black', 'college', 'lowinc', 'age', 'pid7c', 'm4a', 'minwage15', 'abortion_cons', 'econ_perception', 'validated_2020']:
    rows.append({'variable': v, 'imm_cons_surge': wmean(sci[v], sci['wt']), 'other_surge': wmean(scl[v], scl['wt'])})
surge_split = pd.DataFrame(rows).set_index('variable')
print(f'\nWithin the surge cohort: immigration-conservative (N = {len(sci):,}) vs the rest (N = {len(scl):,}):'); print(surge_split.round(2).to_string())
print('\nThe immigration-conservative surge voter was more often Black, older, more abortion-conservative, and rated Trump\'s 2020')
print('economy less harshly (-0.99 vs -1.40) - while supporting Medicare-for-all at 86%.')

## 3. CES 2024: where the switchers came from

In [ ]:
cols = ['TS_g2024', 'vvweight_post', 'vvweight', 'presvote20post', 'presvote16post', 'CC24_410', 'pid7', 'race', 'hispanic',
        'CC24_323a', 'CC24_323b', 'CC24_323c', 'CC24_340f', 'CC24_323d', 'CC24_328e']
d = pd.read_csv(RAW_DIR / 'ces' / 'CCES24_Common_OUTPUT_vv_topost_final.csv', usecols=cols, low_memory=False)
d['wt'] = d['vvweight_post'].fillna(d['vvweight']); d['voted24'] = d['TS_g2024'].notna() & (d['TS_g2024'] != 7)
d['v20'] = d['presvote20post'].map({1: 'Biden', 2: 'Trump', 6: 'nonvoter'}); d['v16'] = d['presvote16post'].map({1: 'Clinton', 2: 'Trump', 7: 'nonvoter'}); d['v24'] = d['CC24_410'].map({1: 'Harris', 2: 'Trump'})
cons = lambda c, sup: (d[c] == (1 if sup else 2)).astype(float).where(d[c].notna())
d['imm_cons'] = pd.concat([cons('CC24_323a', False), cons('CC24_323b', True), cons('CC24_323c', True), cons('CC24_340f', True), cons('CC24_323d', False)], axis=1).mean(axis=1)
d['immC'] = (d['imm_cons'] >= 0.5).astype(float); d['expand_medicaid'] = cons('CC24_328e', True)
d['hisp'] = (d['hispanic'] == 1).astype(float); d['black'] = ((d['race'] == 2) & (d['hispanic'] != 1)).astype(float); d['white'] = ((d['race'] == 1) & (d['hispanic'] != 1)).astype(float)
b24 = d[d['voted24'] & (d['v20'] == 'Biden') & d['v24'].isin(['Harris', 'Trump'])].copy(); b24['switch'] = (b24['v24'] == 'Trump').astype(float)
sw = b24[b24['switch'] == 1]
print(f'CES 2024, recalled-Biden-2020 voters with a validated 2024 vote and 2024 choice: N = {len(b24):,}; switchers = {len(sw)}')
rows = []
for s in ['Clinton', 'nonvoter', 'Trump']:
    g = b24[b24['v16'] == s]
    rows.append({'2016_vote': s, 'N': len(g), 'share_of_Biden20_voters': g['wt'].sum() / b24['wt'].sum(), 'share_of_switchers': wmean((sw['v16'] == s).astype(float), sw['wt']),
                 'imm_cons_share': wmean(g['immC'], g['wt']), 'switch_rate': wmean(g['switch'], g['wt']),
                 'switch_if_immcons': wmean(g.loc[g['immC'] == 1, 'switch'], g.loc[g['immC'] == 1, 'wt']), 'switch_if_not': wmean(g.loc[g['immC'] == 0, 'switch'], g.loc[g['immC'] == 0, 'wt'])})
origin = pd.DataFrame(rows).set_index('2016_vote')
print('\nBy recalled 2016 vote:'); print(origin.round(3).to_string())
print('\nThe panel (notebook 12, 39 switchers) put the Trump-2016 share of switchers at 4% weighted; the cross-section, on 16,807')
print('recalled-Biden voters, puts it at 27%. The cross-section figure supersedes the panel\'s on this point.')

In [ ]:
r = b24[b24['immC'] == 1]; nr = b24[b24['immC'] == 0]
print(f'The reservoir: immigration-conservative Biden-2020 voters = {wmean(b24["immC"], b24["wt"]):.1%} of Biden voters')
print(f'  switch rate {wmean(r["switch"], r["wt"]):.1%}  vs  {wmean(nr["switch"], nr["wt"]):.1%} for the rest')
print(f'  share of all switchers: {wmean(sw["immC"], sw["wt"]):.0%}')
rows = []
for lab, m in [('Hispanic', r['hisp'] == 1), ('Black', r['black'] == 1), ('white', r['white'] == 1), ('ALL', r['immC'] == 1)]:
    g = r[m]; rows.append({'group': lab, 'N': len(g), 'switch_rate': wmean(g['switch'], g['wt']), 'expand_medicaid': wmean(g['expand_medicaid'], g['wt']),
                           'share_of_reservoir': g['wt'].sum() / r['wt'].sum()})
res24 = pd.DataFrame(rows).set_index('group')
print('\nReservoir by race/ethnicity:'); print(res24.round(3).to_string())
print('\nComposition of switchers vs all Biden-2020 voters: Hispanic {:.0%} vs {:.0%}; Black {:.0%} vs {:.0%}; white {:.0%} vs {:.0%}'.format(
    wmean(sw['hisp'], sw['wt']), wmean(b24['hisp'], b24['wt']), wmean(sw['black'], sw['wt']), wmean(b24['black'], b24['wt']), wmean(sw['white'], sw['wt']), wmean(b24['white'], b24['wt'])))

## 3b. Robustness: cutoff, party identification, and pre-switch attitudes

The reservoir is defined on 2024 attitudes, measured after the vote. Three checks: is it sensitive to the 0.5 cutoff; does it survive within levels of party identification (or is it weak partisanship in disguise); and what does the panel say when the reservoir is defined on 2020 attitudes, before anyone had switched?

In [ ]:
b24['pid'] = b24['pid7'].where(b24['pid7'] <= 7)
rows = []
for cut in [0.4, 0.5, 0.6, 0.8]:
    r = b24[b24['imm_cons'] >= cut]; n = b24[b24['imm_cons'] < cut]
    rows.append({'cutoff': cut, 'share_of_Biden20': wmean((b24['imm_cons'] >= cut).astype(float), b24['wt']), 'switch_reservoir': wmean(r['switch'], r['wt']), 'switch_rest': wmean(n['switch'], n['wt']), 'share_of_switchers': wmean((sw['imm_cons'] >= cut).astype(float), sw['wt'])})
cutoffs = pd.DataFrame(rows).set_index('cutoff'); print('Cutoff sensitivity:'); print(cutoffs.round(3).to_string())
rows = []
for lab, m in [('strong Dem (1)', b24['pid'] == 1), ('weak/lean Dem (2-3)', b24['pid'].isin([2, 3])), ('independent (4)', b24['pid'] == 4), ('lean/weak/strong Rep (5-7)', b24['pid'] >= 5)]:
    g = b24[m]; r = g[g['immC'] == 1]; n = g[g['immC'] == 0]
    rows.append({'party_id': lab, 'N': len(g), 'reservoir_share': wmean(g['immC'], g['wt']), 'switch_reservoir': wmean(r['switch'], r['wt']), 'switch_rest': wmean(n['switch'], n['wt'])})
by_pid = pd.DataFrame(rows).set_index('party_id'); print('
Reservoir by party ID:'); print(by_pid.round(3).to_string())
print('
The ratio holds at every level of party ID (10-18x). The 22% average rate is carried by independents and leaners who voted Biden;')
print('among strong Democrats the reservoir switched at 3.7%.')

In [ ]:
wp = pd.read_csv(RAW_DIR / 'ces_panel' / 'merged_recontact_2024_vv.csv', usecols=['commonpostweight_24', 'commonweight_24', 'CC20_410', 'CC24_410', 'CC20_331a', 'CC20_331b', 'CC20_331d', 'CC20_331e', 'CC20_327a'], low_memory=False)
wp['wt'] = wp['commonpostweight_24'].fillna(wp['commonweight_24'])
pc = lambda c, sup: (wp[c] == (1 if sup else 2)).astype(float).where(wp[c].notna())
wp['imm20'] = pd.concat([pc('CC20_331a', False), pc('CC20_331b', True), pc('CC20_331d', True), pc('CC20_331e', True)], axis=1).mean(axis=1)
pb = wp[(wp['CC20_410'] == 1) & wp['CC24_410'].isin([1, 2])].copy(); pb['switch'] = (pb['CC24_410'] == 2).astype(float); psw = pb[pb['switch'] == 1]
rows = []
for cut in [0.5, 0.75]:
    r = pb[pb['imm20'] >= cut]; n = pb[pb['imm20'] < cut]
    rows.append({'cutoff_2020': cut, 'share_of_Biden20': wmean((pb['imm20'] >= cut).astype(float), pb['wt']), 'N_reservoir': len(r), 'switch_reservoir_wtd': wmean(r['switch'], r['wt']), 'switch_rest_wtd': wmean(n['switch'], n['wt']),
                 'switch_reservoir_unwtd': r['switch'].mean(), 'switch_rest_unwtd': n['switch'].mean(), 'events_reservoir': int(r['switch'].sum()), 'events_rest': int(n['switch'].sum()), 'share_of_switchers': wmean((psw['imm20'] >= cut).astype(float), psw['wt'])})
pre = pd.DataFrame(rows).set_index('cutoff_2020'); print(f'Panel: reservoir defined on 2020 attitudes, 2024 outcome (N = {len(pb):,} Biden-2020 voters, {int(pb["switch"].sum())} switchers):'); print(pre.round(3).to_string())
print(f'M4A support (2020) among the 2020-defined reservoir: {wmean(pc("CC20_327a", True)[pb.index][pb["imm20"] >= 0.5], pb.loc[pb["imm20"] >= 0.5, "wt"]):.0%}')
print('
With attitudes measured before the switch, the reservoir is 17% of the coalition, switches at 13.5% vs 2.2%, and supplies 56% of')
print('switchers. The 2024-attitude version (19% / 22% / 81%) includes attitude change that accompanied the switch (notebook 12: +0.24).')

## 4. Save

In [ ]:
prof20.to_csv(TABLES_DIR / 'coalition_2020_by_2016_vote.csv'); res20.to_csv(TABLES_DIR / 'reservoir_2020.csv'); surge_split.to_csv(TABLES_DIR / 'surge_cohort_split.csv')
origin.to_csv(TABLES_DIR / 'switchers_2024_by_2016_vote.csv'); res24.to_csv(TABLES_DIR / 'reservoir_2024.csv'); cutoffs.to_csv(TABLES_DIR / 'reservoir_cutoffs.csv'); by_pid.to_csv(TABLES_DIR / 'reservoir_by_pid.csv'); pre.to_csv(TABLES_DIR / 'reservoir_2020_attitudes_panel.csv')
print('Saved 5 tables to outputs/tables/')

## Limitations

- **2016 vote is recalled** — in 2020 for the 2020 file, in 2024 for the 2024 file. Recall favors the winner and the respondent's current party; a 2024 Trump voter is more likely to recall a 2016 Trump vote. The 27% Trump-2016 share among switchers is likely an upper bound; the panel's 4% is a lower bound on eight people.
- **Immigration indices differ across waves.** 2020 uses four items, 2024 five (adds asylum, drops "cut legal immigration"). Cutoff at 0.5 in both; the reservoir shares (19% in each) are consistent but the items are not identical.
- **2020 economic items are Medicare-for-all and $15 minimum wage; 2024's closest is Medicaid expansion.**
- **The 2024-attitude reservoir overstates pre-switch predictive power.** §3b gives the panel version on 2020 attitudes: a 6x ratio and 56% of switchers, not 18x and 81%. The difference is attitude change concurrent with the switch.